# Main

In [13]:
import helpers.assembly as assembly
import helpers.element as element
import helpers.interfaces as interfaces
import helpers.partition as partition
import helpers.preprocess as preprocess
import numpy as np

In [14]:
nodes = {
    1:  (0.0,    0.0, 0.0),      # B0
    2:  (4.744,  0.0, 0.0),      # B1
    3:  (4.744,  0.0, 6.796),   # T1

    4:  (9.488,  0.0, 0.0),      # B2
    5:  (9.488,  0.0, 7.724),   # T2

    6:  (14.232, 0.0, 0.0),      # B3
    7:  (14.232, 0.0, 8.419),   # T3

    8:  (18.976, 0.0, 0.0),      # B4
    9:  (18.976, 0.0, 8.883),   # T4

    10: (23.720, 0.0, 0.0),      # B5
    11: (23.720, 0.0, 9.115),   # T5

    12: (28.464, 0.0, 0.0),      # B6
    13: (28.464, 0.0, 9.115),   # T6

    14: (33.208, 0.0, 0.0),      # B7
    15: (33.208, 0.0, 8.883),   # T7

    16: (37.952, 0.0, 0.0),      # B8
    17: (37.952, 0.0, 8.419),   # T8

    18: (42.696, 0.0, 0.0),      # B9
    19: (42.696, 0.0, 7.724),   # T9

    20: (47.440, 0.0, 0.0),      # B10
    21: (47.440, 0.0, 6.796),   # T10

    22: (52.184, 0.0, 0.0),      # B11

    23:  (0.0,    10.0, 0.0),      # B0'
    24:  (4.744,  10.0, 0.0),      # B1'
    25:  (4.744,  10.0, 6.796),   # T1'

    26:  (9.488,  10.0, 0.0),      # B2'
    27:  (9.488,  10.0, 7.724),   # T2'

    28:  (14.232, 10.0, 0.0),      # B3'
    29:  (14.232, 10.0, 8.419),   # T3'

    30:  (18.976, 10.0, 0.0),      # B4'
    31:  (18.976, 10.0, 8.883),   # T4'

    32: (23.720, 10.0, 0.0),      # B5'
    33: (23.720, 10.0, 9.115),   # T5'

    34: (28.464, 10.0, 0.0),      # B6'
    35: (28.464, 10.0, 9.115),   # T6'

    36: (33.208, 10.0, 0.0),      # B7'
    37: (33.208, 10.0, 8.883),   # T7'

    38: (37.952, 10.0, 0.0),      # B8'
    39: (37.952, 10.0, 8.419),   # T8'

    40: (42.696, 10.0, 0.0),      # B9'
    41: (42.696, 10.0, 7.724),   # T9'

    42: (47.440, 10.0, 0.0),      # B10'
    43: (47.440, 10.0, 6.796),   # T10'

    44: (52.184, 10.0, 0.0),      # B11'
}

In [15]:
filepath = "inputs/validation_truss.json"

nodes = interfaces.read_nodes(filepath)
elements = interfaces.read_elements(filepath)
constraints = interfaces.read_constraints(filepath)
materials = interfaces.read_materials(filepath)
sections = interfaces.read_sections(filepath)
constraints = interfaces.read_constraints(filepath)
nodal_loads = interfaces.read_nodal_loads(filepath)
element_loads = interfaces.read_element_loads(filepath)
temperature_loads = interfaces.read_temperature_loads(filepath)
fabrication_loads = interfaces.read_fabrication_errors(filepath)

print(element_loads)


{1: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 2: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}


In [16]:
k_list = []
T_list = []
Qf_list = []
map_list = []

element_paras = element.build_elements_para(nodes, elements, materials, sections)

for e_id in elements:
        L, etype, E, G, A, Iy, Iz, J, mt = element_paras[e_id]
        kl = element.element_kl(E, G, A, Iy, Iz, J, L)
        gamma = element.build_gamma_3d(nodes[elements[e_id][3][0]], nodes[elements[e_id][3][1]], psi=0.0, angle_unit="deg")
        T = element.frame_transformation_matrix(gamma)
        Qf = preprocess.fef_cal(element_loads.get(e_id), L, angle_unit="deg")
        m = element.element_dof_map_1based(elements[e_id][3][0], elements[e_id][3][1])
        print(m)


        k_list.append(kl)
        T_list.append(T)
        Qf_list.append(Qf)
        map_list.append(m)

ndof = int(np.max(np.concatenate(map_list)))


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
[7, 8, 9, 10, 11, 12, 19, 20, 21, 22, 23, 24]
[19, 20, 21, 22, 23, 24, 31, 32, 33, 34, 35, 36]
[31, 32, 33, 34, 35, 36, 43, 44, 45, 46, 47, 48]
[43, 44, 45, 46, 47, 48, 55, 56, 57, 58, 59, 60]
[55, 56, 57, 58, 59, 60, 67, 68, 69, 70, 71, 72]
[67, 68, 69, 70, 71, 72, 79, 80, 81, 82, 83, 84]
[79, 80, 81, 82, 83, 84, 91, 92, 93, 94, 95, 96]
[91, 92, 93, 94, 95, 96, 103, 104, 105, 106, 107, 108]
[103, 104, 105, 106, 107, 108, 115, 116, 117, 118, 119, 120]
[115, 116, 117, 118, 119, 120, 127, 128, 129, 130, 131, 132]
[13, 14, 15, 16, 17, 18, 25, 26, 27, 28, 29, 30]
[25, 26, 27, 28, 29, 30, 37, 38, 39, 40, 41, 42]
[37, 38, 39, 40, 41, 42, 49, 50, 51, 52, 53, 54]
[49, 50, 51, 52, 53, 54, 61, 62, 63, 64, 65, 66]
[61, 62, 63, 64, 65, 66, 73, 74, 75, 76, 77, 78]
[73, 74, 75, 76, 77, 78, 85, 86, 87, 88, 89, 90]
[85, 86, 87, 88, 89, 90, 97, 98, 99, 100, 101, 102]
[97, 98, 99, 100, 101, 102, 109, 110, 111, 112, 113, 114]
[109, 110, 111, 112, 113, 114, 121, 122